In [1]:
import os
from dotenv import load_dotenv
load_dotenv("../.env") 

## CONFIG
LLM_BASE_URL=os.getenv("LLM_BASE_URL")
LLM_API_KEY=os.getenv("LLM_API_KEY")
LLM_MODEL_NAME=os.getenv("LLM_MODEL_NAME")

QDRANT_API_KEY=os.getenv("QDRANT_API_KEY")
QDRANT_ENDPOINT=os.getenv("QDRANT_ENDPOINT")
QDRANT_COLLECTION_NAME=os.getenv("QDRANT_COLLECTION_NAME")

In [2]:
from pathlib import Path

In [3]:
PDFs_DIR = Path("./data")
PDFs_PATH = [str(i).lower() for i in sorted(PDFs_DIR.glob("*.pdf"))]
print(PDFs_PATH)

['data/cv-hansnaufalgranito.pdf', 'data/mbti-h.pdf']


In [4]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
chunks = []

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

for pdf_path in PDFs_PATH:
    pages = PyPDFLoader(pdf_path).load()
    chunks.extend(text_splitter.split_documents(pages))

In [13]:
len(chunks)

66

## Start Embed

In [14]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

In [15]:
qdrant_client = QdrantClient(
    url=QDRANT_ENDPOINT,
    api_key=QDRANT_API_KEY
)

/var/folders/n4/7np7n0m17ds0d81wpq0ln_kr0000gn/T/ipykernel_6912/1595527835.py:1: UserWarning: Api key is used with an insecure connection.
  qdrant_client = QdrantClient(


/Users/kaktus/miniconda3/envs/faysrag/lib/python3.12/site-packages/qdrant_client/qdrant_remote.py:282: UserWarning: Qdrant client version 1.18.0 is incompatible with server version 1.20.2. Major versions should match and minor version difference must not exceed 1. Set check_compatibility=False to skip version check.
  show_warning(


In [16]:
existing = [c.name for c in qdrant_client.get_collections().collections]
if QDRANT_COLLECTION_NAME in existing:
    qdrant_client.delete_collection(QDRANT_COLLECTION_NAME)
    print(f"Dropped '{QDRANT_COLLECTION_NAME}'")
    
qdrant_client.create_collection(
    collection_name=QDRANT_COLLECTION_NAME,
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)
print(f"Created '{QDRANT_COLLECTION_NAME}'")

Created 'nyeh-agentik-coll'


## Embedding

In [17]:
import openai
import uuid

In [18]:
openai_client = openai.Client(
    api_key=LLM_API_KEY,
    base_url=LLM_BASE_URL,
)

In [19]:
for i, chunk in enumerate(chunks):
    response = openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=chunk.page_content,
    )

    point = PointStruct(
        id=str(uuid.uuid5(uuid.NAMESPACE_URL, f"{chunk.metadata['source']}:{chunk.page_content}")),
        vector=response.data[0].embedding,
        payload={
            "text": chunk.page_content,
            "metadata": chunk.metadata,
        },
    )

    qdrant_client.upsert(
        collection_name=QDRANT_COLLECTION_NAME,
        points=[point],
        timeout=30,
    )
    
    print(f"Upserted {i+1}/{len(chunks)}")

Upserted 1/66
Upserted 2/66
Upserted 3/66
Upserted 4/66
Upserted 5/66
Upserted 6/66
Upserted 7/66
Upserted 8/66
Upserted 9/66
Upserted 10/66
Upserted 11/66
Upserted 12/66
Upserted 13/66
Upserted 14/66
Upserted 15/66
Upserted 16/66
Upserted 17/66
Upserted 18/66
Upserted 19/66
Upserted 20/66
Upserted 21/66
Upserted 22/66
Upserted 23/66
Upserted 24/66
Upserted 25/66
Upserted 26/66
Upserted 27/66
Upserted 28/66
Upserted 29/66
Upserted 30/66
Upserted 31/66
Upserted 32/66
Upserted 33/66
Upserted 34/66
Upserted 35/66
Upserted 36/66
Upserted 37/66
Upserted 38/66
Upserted 39/66
Upserted 40/66
Upserted 41/66
Upserted 42/66
Upserted 43/66
Upserted 44/66
Upserted 45/66
Upserted 46/66
Upserted 47/66
Upserted 48/66
Upserted 49/66
Upserted 50/66
Upserted 51/66
Upserted 52/66
Upserted 53/66
Upserted 54/66
Upserted 55/66
Upserted 56/66
Upserted 57/66
Upserted 58/66
Upserted 59/66
Upserted 60/66
Upserted 61/66
Upserted 62/66
Upserted 63/66
Upserted 64/66
Upserted 65/66
Upserted 66/66


In [20]:
# test

def retrieval(msg):
    vec =  openai_client.embeddings.create(
        model="text-embedding-3-small",
        input=msg,
    ).data[0].embedding
    
    retrieved = qdrant_client.query_points(
        collection_name=QDRANT_COLLECTION_NAME,
        query=vec,
        limit=5,
    )
    
    return retrieved.points

In [21]:
retrieval("Halo, Nama kamu siapa?")

[ScoredPoint(id='43a897e5-1d80-5c93-9abc-acd963ed8f77', version=1, score=0.26147637, payload={'text': 'HANS NAUFAL GRANITO\nMachine Learning Engineer — GenAI/ML & Cloud-Native Systems\n+62 812-1805-5705 | hansnaufalgranito@gmail.com | linkedin.com/in/hansnaufalgranito | github.com/hng011 | South Tangerang, Indonesia\nMachine Learning Engineer shipping production GenAI systems — agentic/RAG, NL2SQL, and LLM-OCR pipelines — on GCP and \nAzure for five enterprise clients across mining, fintech, retail, and agribusiness. Depth in LLM security guardrails, Vertex AI inference \noptimization, and cloud-native backend architecture. Published researcher (peer-reviewed, 2025), BNSP-certified Associate Data \nScientist, and certified TensorFlow Developer (DCML); graduated 3.90/4.00 GPA.\nPROFESSIONAL EXPERIENCE\nData Labs Analytics (datalabs.id) Oct 2025 – Present\nMachine Learning Engineer\n• Architected production RAG and multi-agent GenAI systems (Google ADK, LangChain, LangGraph, Vertex AI, B